# ControlledRAG optional modern judge — repaired build-only notebook

**Status: PREPARED_NOT_EXECUTED.** The default path runs only synthetic tests. Real execution requires two separately edited `True` gates, a complete immutable config, and a later explicit author authorization.

In [ ]:
from pathlib import Path
import csv, importlib.metadata, importlib.util, json, os, subprocess, sys

RUN_REAL_INFERENCE = False
PACKAGE_RESULT_ZIP = False
PACKAGE_DIR = Path.cwd()
CONFIG_PATH = PACKAGE_DIR / 'run_config.json'
print({'RUN_REAL_INFERENCE': RUN_REAL_INFERENCE, 'PACKAGE_RESULT_ZIP': PACKAGE_RESULT_ZIP, 'package_dir': str(PACKAGE_DIR)})

In [ ]:
# Default path: zero network calls, zero model loads, zero real rows.
if not RUN_REAL_INFERENCE:
    completed = subprocess.run(
        [sys.executable, str(PACKAGE_DIR / 'CPU_SYNTHETIC_SMOKE_TEST.py')],
        cwd=PACKAGE_DIR,
        check=True,
        capture_output=True,
        text=True,
        env={**os.environ, 'CONTROLLEDRAG_NOTEBOOK_CHILD': '1'},
    )
    print(completed.stdout.strip())
    print('Synthetic repair suite complete. Real execution remains disabled.')

In [ ]:
# Separate config gate, dependency preflight, T4x2 detection, and strict validation.
if RUN_REAL_INFERENCE:
    from errors import ConfigurationError
    from huggingface_adapter import detect_accelerators
    from run_config import load_and_validate_config
    preliminary = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
    if preliminary.get('run_real_inference') is not True:
        raise ConfigurationError('both notebook and run-config real-run gates must be true')
    required_modules = ['torch', 'transformers', 'jsonschema']
    if preliminary.get('quantization') in {'4bit', '8bit'}:
        required_modules.append('bitsandbytes')
    missing_dependencies = [name for name in required_modules if importlib.util.find_spec(name) is None]
    if missing_dependencies:
        raise ConfigurationError(f'missing frozen runtime dependencies: {missing_dependencies}')
    topology = detect_accelerators()
    device_names = tuple(str(device['name']) for device in topology['devices'])
    config = load_and_validate_config(CONFIG_PATH, device_names=device_names)
    t4x2_detected = topology['gpu_count'] == 2 and all('T4' in name.upper() for name in device_names)
    if not t4x2_detected:
        raise ConfigurationError('this notebook requires the declared Kaggle T4x2 topology')
    if config['model_source_mode'] == 'offline_snapshot' and not Path(config['model_snapshot_path']).is_dir():
        raise ConfigurationError('uploaded immutable model snapshot directory is missing')
else:
    config = None
    topology = None

In [ ]:
# Frozen source/manifest verification, canonical prompt rendering, and resume-state loading.
if RUN_REAL_INFERENCE:
    from errors import IntegrityError
    from prompt_rendering import render_prompt
    from run_state import load_attempts, load_checkpoint, reconcile_attempt_snapshots, terminal_state, atomic_write_text
    from SAMPLE_SELECTION import EXPECTED_SOURCE_SHA256, IDENTITY_FIELDS, canonical_digest, sha256_file
    source_path = Path(config['source_csv'])
    manifest_path = PACKAGE_DIR / config['candidate_manifest']
    if config['source_sha256'] != EXPECTED_SOURCE_SHA256 or sha256_file(source_path) != EXPECTED_SOURCE_SHA256:
        raise IntegrityError('fixed source SHA mismatch')
    candidate_index = json.loads((PACKAGE_DIR / 'SAMPLE_MANIFEST_CANDIDATES' / 'candidates_index.json').read_text(encoding='utf-8'))
    committed_hashes = {record['file']: record['sha256'] for record in candidate_index['candidate_records']}
    if committed_hashes.get(manifest_path.name) != config['candidate_manifest_sha256'] or sha256_file(manifest_path) != config['candidate_manifest_sha256']:
        raise IntegrityError('candidate manifest SHA mismatch')
    with source_path.open(newline='', encoding='utf-8') as handle:
        source_rows = list(csv.DictReader(handle))
    with manifest_path.open(newline='', encoding='utf-8') as handle:
        manifest_rows = list(csv.DictReader(handle))
    if len({row['row_id'] for row in manifest_rows}) != len(manifest_rows):
        raise IntegrityError('duplicate candidate row_id')
    joined_rows, rendered_by_row = [], {}
    for manifest_row in manifest_rows:
        source = source_rows[int(manifest_row['source_row_index'])]
        if canonical_digest(source, IDENTITY_FIELDS) != manifest_row['input_digest']:
            raise IntegrityError(f'input digest mismatch: {manifest_row["row_id"]}')
        joined = {**manifest_row, 'question': source['question'], 'context': source['context'], 'answer': source['answer']}
        joined_rows.append(joined)
        rendered_by_row[manifest_row['row_id']] = render_prompt(PACKAGE_DIR / 'PROMPT_TEMPLATE.md', question=source['question'], context=source['context'], answer=source['answer'], transport=config['prompt_transport'])
    result_path = PACKAGE_DIR / config['output_jsonl']
    attempt_path = PACKAGE_DIR / config['attempt_log_jsonl']
    checkpoint_path = PACKAGE_DIR / config['checkpoint_json']
    result_dir = result_path.parent
    scratch_dir = result_dir / 'scratch'
    scratch_dir.mkdir(parents=True, exist_ok=True)
    atomic_write_text(result_dir / 'run_config.json', CONFIG_PATH.read_text(encoding='utf-8'))
    atomic_write_text(result_dir / 'candidate_manifest.csv', manifest_path.read_text(encoding='utf-8'))
    attempt_groups = []
    if attempt_path.exists(): attempt_groups.append(load_attempts(attempt_path))
    for shard_path in sorted(scratch_dir.glob('shard_*_attempts.jsonl')):
        attempt_groups.append(load_attempts(shard_path))
    existing_attempts = reconcile_attempt_snapshots(*attempt_groups) if attempt_groups else []
    if existing_attempts and not config['resume']:
        raise IntegrityError('existing attempts found while resume=false')
    checkpoint = load_checkpoint(checkpoint_path)
    if checkpoint and checkpoint['run_id'] != config['run_id']:
        raise IntegrityError('checkpoint run_id mismatch')
    existing_by_row = {}
    for record in existing_attempts:
        existing_by_row.setdefault(record['row_id'], []).append(record)
    pending_rows = []
    for row in joined_rows:
        prior = existing_by_row.get(row['row_id'], [])
        if not prior or not terminal_state(prior[-1], attempt_count=len(prior)):
            pending_rows.append(row)
    print({'candidate_rows': len(joined_rows), 'stored_attempts': len(existing_attempts), 'pending_rows': len(pending_rows)})

In [ ]:
# Real execution uses shared retry/error/schema contracts; shard attempts are append-only.
if RUN_REAL_INFERENCE:
    from concurrent.futures import ThreadPoolExecutor
    from huggingface_adapter import HuggingFaceAdapter
    from judge_execution import execute_row_with_retries
    from provider_adapter import RoutingPolicy
    from run_state import append_attempt, atomic_write_json, atomic_write_jsonl, derive_final_terminal_records
    policy = RoutingPolicy(config['provider'], config['model'], config['model_revision'], tuple(config['allowed_returned_models']), tuple(config['allowed_routed_via']), config['allow_fallback_attempts'])
    def run_hf_shard(rows, device_index):
        adapter = HuggingFaceAdapter(policy=policy, model_id=config['model_id'], revision=config['model_revision'], model_source_mode=config['model_source_mode'], model_snapshot_path=config['model_snapshot_path'], quantization=config['quantization'], dtype=config['dtype'], gpu_strategy=config['gpu_strategy'], device_index=device_index, local_files_only=config['local_files_only'], prompt_transport=config['prompt_transport'], model_context_window=config['model_context_window'], max_new_tokens=config['max_new_tokens'], generation_seed=config['generation_seed'], allow_model_load=True)
        shard_attempt_path = scratch_dir / f'shard_{device_index}_attempts.jsonl'
        shard_checkpoint_path = scratch_dir / f'shard_{device_index}_checkpoint.json'
        emitted = []
        for row_index, row in enumerate(rows, start=1):
            new_records = execute_row_with_retries(adapter=adapter, policy=policy, row=row, rendered_prompt=rendered_by_row[row['row_id']], run_id=config['run_id'], max_attempts=config['max_attempts_per_row'], existing_attempts=existing_by_row.get(row['row_id'], []), attempt_sink=lambda record: append_attempt(shard_attempt_path, record), device_assignment=f'cuda:{device_index}' if config['gpu_strategy'] == 'one_worker_per_gpu' else 'device_map:auto', dtype=config['dtype'], quantization=config['quantization'])
            emitted.extend(new_records)
            if row_index % config['checkpoint_every'] == 0 or row_index == len(rows):
                atomic_write_json(shard_checkpoint_path, {'run_id': config['run_id'], 'device_index': device_index, 'rows_seen': row_index, 'attempt_keys': [[record['run_id'], record['row_id'], record['attempt']] for record in emitted]})
        return emitted
    if config['gpu_strategy'] == 'one_worker_per_gpu':
        shards = [pending_rows[index::2] for index in range(2)]
        with ThreadPoolExecutor(max_workers=2) as pool:
            new_groups = list(pool.map(run_hf_shard, shards, range(2)))
    else:
        new_groups = [run_hf_shard(pending_rows, 0)]
    refreshed_groups = []
    if attempt_path.exists(): refreshed_groups.append(load_attempts(attempt_path))
    for shard_path in sorted(scratch_dir.glob('shard_*_attempts.jsonl')):
        refreshed_groups.append(load_attempts(shard_path))
    merged_attempts = reconcile_attempt_snapshots(*refreshed_groups) if refreshed_groups else []
    finals = derive_final_terminal_records(merged_attempts)
    unknown_final_ids = sorted(set(finals) - {row['row_id'] for row in manifest_rows})
    if unknown_final_ids:
        raise IntegrityError(f'final rows outside manifest: {unknown_final_ids[:3]}')
    ordered_finals = [finals[row['row_id']] for row in manifest_rows if row['row_id'] in finals]
    atomic_write_jsonl(attempt_path, merged_attempts)
    atomic_write_jsonl(result_path, ordered_finals)
    pending_ids = [row['row_id'] for row in manifest_rows if row['row_id'] not in finals]
    atomic_write_json(checkpoint_path, {'run_id': config['run_id'], 'terminal_row_ids': sorted(finals), 'pending_row_ids': pending_ids, 'attempt_count': len(merged_attempts)})
    runtime_metadata = {'run_id': config['run_id'], 'topology': topology, 't4x2_detected': t4x2_detected, 'gpu_strategy': config['gpu_strategy'], 'requested_dtype': config['dtype'], 'requested_quantization': config['quantization'], 'prompt_transport': config['prompt_transport'], 'model_source_mode': config['model_source_mode'], 'model_revision': config['model_revision'], 'context_length_policy': config['context_length_policy'], 'do_sample': False, 'temperature': 0.0, 'top_p': 1.0, 'top_k': 0, 'max_new_tokens': config['max_new_tokens'], 'generation_seed': config['generation_seed'], 'python_version': sys.version.split()[0], 'torch_version': importlib.metadata.version('torch'), 'transformers_version': importlib.metadata.version('transformers')}
    atomic_write_json(result_dir / 'runtime_metadata.json', runtime_metadata)
    print({'terminal_rows': len(finals), 'pending_rows': len(pending_ids), 'attempts': len(merged_attempts), 'routing_policy': config['routing_rejection_policy']})

In [ ]:
# Package only after the separate strict post-run analysis passes.
if RUN_REAL_INFERENCE and PACKAGE_RESULT_ZIP:
    analysis_path = result_path.parent / 'post_run_analysis.json'
    analysis = json.loads(analysis_path.read_text(encoding='utf-8'))
    if analysis.get('scientifically_valid') is not True:
        raise RuntimeError('strict post-run gates did not pass')
    subprocess.run([sys.executable, str(PACKAGE_DIR / 'BUILD_RESULT_ZIP.py'), '--input-dir', str(result_path.parent), '--output-zip', str(PACKAGE_DIR / 'controlledrag_modern_judge_results.zip')], check=True)
elif RUN_REAL_INFERENCE:
    print('Run POST_RUN_ANALYSIS.py first. The optional run remains quarantined unless scientifically_valid=true.')